In [1]:
import pandas as pd
from collections import defaultdict

In [2]:
matches = pd.read_csv('../data/processed/features_v2.csv')

matches['MatchDateTime'] = pd.to_datetime(matches['MatchDateTime'])

matches = matches.sort_values('MatchDateTime').reset_index(drop=True)

matches.head()

,Season,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,...,HomeGoalsPerGame,AwayGoalsPerGame,HomeGoalsAgainstPerGame,AwayGoalsAgainstPerGame,HomeGDPerGame,AwayGDPerGame,PPGDiff,GDPerGameDiff,GoalsPerGameDiff,GoalsAgainstPerGameDiff
0,21-22,E0,13/08/2021,20:00,Brentford,Arsenal,2,0,H,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,21-22,E0,14/08/2021,12:30,Man United,Leeds,5,1,H,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,21-22,E0,14/08/2021,15:00,Burnley,Brighton,1,2,A,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,21-22,E0,14/08/2021,15:00,Chelsea,Crystal Palace,3,0,H,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,21-22,E0,14/08/2021,15:00,Everton,Southampton,3,1,H,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
home_home_points_last5 = []
home_home_goals_last5 = []
home_home_goals_against_last5 = []

away_away_points_last5 = []
away_away_goals_last5 = []
away_away_goals_against_last5 = []

In [4]:
home_history = defaultdict(list)
away_history = defaultdict(list)

In [5]:
def calc_home_form(previous_home_matches):
    if len(previous_home_matches) == 0:
        return 0, 0, 0

    last5 = previous_home_matches[-5:]

    points = 0
    goals_for = 0
    goals_against = 0

    for match in last5:
        goals_for += match['FTHG']
        goals_against += match['FTAG']

        if match['FTR'] == 'H':
            points += 3
        elif match['FTR'] == 'D':
            points += 1

    games = len(last5)

    return (
        points / games,
        goals_for / games,
        goals_against / games
    )

In [6]:
def calc_away_form(previous_away_matches):
    if len(previous_away_matches) == 0:
        return 0, 0, 0

    last5 = previous_away_matches[-5:]

    points = 0
    goals_for = 0
    goals_against = 0

    for match in last5:
        goals_for += match['FTAG']
        goals_against += match['FTHG']

        if match['FTR'] == 'A':
            points += 3
        elif match['FTR'] == 'D':
            points += 1

    games = len(last5)

    return (
        points / games,
        goals_for / games,
        goals_against / games
    )

In [7]:
for _, current_match in matches.iterrows():

    home_team = current_match['HomeTeam']
    away_team = current_match['AwayTeam']

    # Previous venue-specific matches
    prev_home_matches = home_history[home_team]
    prev_away_matches = away_history[away_team]

    # Calculate features BEFORE current match
    h_points, h_goals, h_against = calc_home_form(prev_home_matches)
    a_points, a_goals, a_against = calc_away_form(prev_away_matches)

    # Store
    home_home_points_last5.append(h_points)
    home_home_goals_last5.append(h_goals)
    home_home_goals_against_last5.append(h_against)

    away_away_points_last5.append(a_points)
    away_away_goals_last5.append(a_goals)
    away_away_goals_against_last5.append(a_against)

    # Add current match AFTER feature calculation
    home_history[home_team].append(current_match)
    away_history[away_team].append(current_match)

In [8]:
matches['HomeHomePointsLast5'] = home_home_points_last5
matches['HomeHomeGoalsLast5'] = home_home_goals_last5
matches['HomeHomeGoalsAgainstLast5'] = home_home_goals_against_last5

matches['AwayAwayPointsLast5'] = away_away_points_last5
matches['AwayAwayGoalsLast5'] = away_away_goals_last5
matches['AwayAwayGoalsAgainstLast5'] = away_away_goals_against_last5

C:\Users\harry\AppData\Local\Temp\ipykernel_28516\2170545325.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['HomeHomePointsLast5'] = home_home_points_last5
C:\Users\harry\AppData\Local\Temp\ipykernel_28516\2170545325.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['HomeHomeGoalsLast5'] = home_home_goals_last5
C:\Users\harry\AppData\Local\Temp\ipykernel_28516\2170545325.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poo

In [9]:
matches['HomeAwayPointsDiffLast5'] = (
    matches['HomeHomePointsLast5'] -
    matches['AwayAwayPointsLast5']
)

matches['HomeAwayGoalsDiffLast5'] = (
    matches['HomeHomeGoalsLast5'] -
    matches['AwayAwayGoalsLast5']
)

# Positive = home team has better venue-specific defence
matches['HomeAwayGoalsAgainstDiffLast5'] = (
    matches['AwayAwayGoalsAgainstLast5'] -
    matches['HomeHomeGoalsAgainstLast5']
)

C:\Users\harry\AppData\Local\Temp\ipykernel_28516\2316315949.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['HomeAwayPointsDiffLast5'] = (
C:\Users\harry\AppData\Local\Temp\ipykernel_28516\2316315949.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['HomeAwayGoalsDiffLast5'] = (
C:\Users\harry\AppData\Local\Temp\ipykernel_28516\2316315949.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining

In [10]:
matches[[
    'HomeTeam',
    'AwayTeam',
    'HomeHomePointsLast5',
    'AwayAwayPointsLast5',
    'HomeAwayPointsDiffLast5'
]].tail(10)

,HomeTeam,AwayTeam,HomeHomePointsLast5,AwayAwayPointsLast5,HomeAwayPointsDiffLast5
1890,Man City,Aston Villa,2.6,0.4,2.2
1891,Sunderland,Chelsea,0.8,0.8,0.0
1892,Nott'm Forest,Bournemouth,1.2,2.2,-1.0
1893,Fulham,Newcastle,1.8,0.8,1.0
1894,Liverpool,Brentford,2.2,1.0,1.2
1895,Brighton,Man United,2.4,1.6,0.8
1896,Crystal Palace,Arsenal,1.8,2.0,-0.2
1897,Burnley,Wolves,0.4,0.2,0.2
1898,Tottenham,Everton,0.4,1.0,-0.6
1899,West Ham,Leeds,1.6,1.4,0.2


In [11]:
matches.groupby('FTR')['HomeAwayPointsDiffLast5'].mean()

FTR
A    0.047968
D    0.303891
H    0.625308
Name: HomeAwayPointsDiffLast5, dtype: float64

In [12]:
matches.to_csv('../data/processed/features_v3.csv', index=False)

print('Saved features_v3.csv')

Saved features_v3.csv
